# CNN–BiLSTM Walk-Forward + Optuna Bayesian Optimization Pipeline

This notebook follows the pipeline you asked for:

- load a cleaned dataframe
- set `HORIZON = 1`
- split once into `df_train_outer = 80%` and `df_test = 20%`
- set one fixed seed for Optuna tuning reproducibility
- run walk-forward validation only on `df_train_outer`
- in every Optuna trial and every fold:
  - define `fold_train_raw` and `fold_val_raw`
  - fit scaler on `fold_train_raw` only
  - transform `fold_val_raw` using that fitted scaler
  - create sliding-window sequences
  - train the model
  - compute fold RMSE
- aggregate fold RMSE as the Optuna objective
- minimize `mean_rmse` directly
- refit preprocessing on full `df_train_outer`
- transform full `df_train_outer` and `df_test`
- create final train/test sequences
- for each seed in `[0, 1, 2, 42, 99, 123]`:
  - set random seed
  - train final model on full `df_train_outer`
  - use last 10% of outer-train as internal time-based validation
  - evaluate on untouched `df_test`
  - store RMSE, R², loss, and val_loss
- report:
  - per-fold RMSE during Optuna walk-forward tuning
  - every per-seed epoch loss and val_loss in text
  - every per-seed RMSE and R² on test
  - mean ± standard deviation across seeds
- plot:
  - every per-seed loss vs val_loss
  - every per-seed predicted vs actual on `df_test` with RMSE and R² in the title


## Notes

This notebook is based on the structure of your uploaded Python notebook/script, but rewritten into a cleaner **single-dataset**, **outer-train/test**, **walk-forward-inside-Optuna**, **multi-seed final evaluation** flow. It removes the old maximize/negative-RMSE pattern and now minimizes `mean_rmse` directly through Optuna.


Updated logic in this version:
- final fit/validation split is done before scaler fitting for leakage-free epoch selection
- final test sequences use train-tail context from outer-train
- after best_epoch is chosen, the model is refit from scratch on the full outer-train before test evaluation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install optuna tensorflow scikit-learn pandas numpy matplotlib


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dropout,
    LSTM,
    Bidirectional,
    Dense,
    BatchNormalization,
    SpatialDropout1D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import optuna
from optuna.samplers import TPESampler

from tensorflow.keras.regularizers import l2


In [ ]:
# =========================
# 2) User Inputs
# =========================
# Replace this path with your actual cleaned dataframe file.
CLEAN_DF_PATH = "/content/drive/MyDrive/ResearchTestingGold/Silver gold Research/Silver gold/CleanDATA_2026_study_aligned/gold_RRL_interpolate.csv"

# Set your target column here.
TARGET_COL = "Gold_Futures"

# Optional: if your file already has a date column, set it here.
DATE_COL = "Date"

# Forecast setting
HORIZON = 1

# Outer split
OUTER_TRAIN_RATIO = 0.80

# Internal validation from the tail of outer-train for final seed runs
FINAL_VAL_RATIO_WITHIN_OUTER_TRAIN = 0.10

# Optuna tuning settings
TUNING_SEED = 42
N_SPLITS_WALK_FORWARD = 5
INIT_POINTS = 15
N_ITER = 40

# Final seed evaluation
FINAL_SEEDS = [0, 1, 2, 42, 99, 123]

# Training settings
MAX_EPOCHS_TUNING = 50
MAX_EPOCHS_FINAL = 50
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4

# Save settings
SAVE_ALL_SEED_MODELS = True
SAVE_ROOT_DIR = "/content/drive/MyDrive/Silver gold Research/Silver gold/ CleanDATA_2026_interpolate/gold_yahoo_all_seed_models_causalcnn_stackedbilstm_0407(10pm)"
REPORTS_DIR = "/content/drive/MyDrive/Silver gold Research/Silver gold/ CleanDATA_2026_interpolate/gold_yahoo_saved_reports_causalcnn_stackedbilstm_0407(10pm)"

PBOUNDS = {
    "lookback": (10, 18),
    "filters": (16, 40),
    "kernel_size": (2, 4),
    "lstm_units": (24, 56),
    "dense_units": (8, 24),
    "dropout_rate": (0.12, 0.22),
    "learning_rate": (8.0e-5, 2.2e-4),
    "l2_reg": (1.0e-6, 5.0e-4),
    "batch_size_exp": (4, 4),  # fixed batch size = 16
}


In [ ]:
# =========================
# 3) Reproducibility Helper
# =========================
def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [ ]:
# =========================
# 4) Load Cleaned DataFrame
# =========================
df = pd.read_csv(CLEAN_DF_PATH)

# --- Validate target first
if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL '{TARGET_COL}' is not present in the dataframe.")

# --- Handle date column properly
if DATE_COL is not None and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    df = df.set_index(DATE_COL)

# --- Ensure target is numeric
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

# --- Keep only numeric columns for modeling
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if TARGET_COL not in numeric_cols:
    raise ValueError(f"TARGET_COL '{TARGET_COL}' must be numeric and present in the dataframe.")

df = df[numeric_cols].copy()

# --- Horizon alignment: predict t+1 using current/past information only
MODEL_TARGET_COL = "target_t_plus_1"
df[MODEL_TARGET_COL] = df[TARGET_COL].shift(-HORIZON)

# --- Remove inf/-inf first
df = df.replace([np.inf, -np.inf], np.nan)

# --- Drop rows with invalid target after shifting
df = df.dropna(subset=[MODEL_TARGET_COL]).copy()

print("Cleaned working dataframe shape:", df.shape)
display(df.head())
display(df.tail())

print("\nNaN count per column:")
print(df.isna().sum().sort_values(ascending=False).head(20))

Cleaned working dataframe shape: (2561, 9)


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,UST10Y_Treasury_Yield,Federal_Funds_Rate,Employment_Pop_Ratio,gepu,gpr_daily,target_t_plus_1
Date,,,,,,,,,
2015-04-01,1208.2,17.059,50.09,1.859,0.12,59.300000,100.978142,138.928131,1200.9
2015-04-02,1200.9,16.701,49.14,1.913,0.12,59.303333,101.094274,113.846565,1218.6
2015-04-06,1218.6,17.110,52.14,1.899,0.12,59.316667,101.558802,116.789253,1210.6
2015-04-07,1210.6,16.840,53.98,1.887,0.12,59.320000,101.674934,110.125252,1203.1
2015-04-08,1203.1,16.454,50.42,1.906,0.12,59.323333,101.791067,166.755096,1193.6


,Gold_Futures,Silver_Futures,Crude_Oil_Futures,UST10Y_Treasury_Yield,Federal_Funds_Rate,Employment_Pop_Ratio,gepu,gpr_daily,target_t_plus_1
Date,,,,,,,,,
2025-04-23,3294.100000,33.547000,62.27,4.384,4.33,60.0,623.352045,146.833679,3348.600000
2025-04-24,3348.600000,33.503000,62.79,4.314,4.33,60.0,623.352045,179.527267,3282.399902
2025-04-25,3282.399902,32.988998,63.02,4.254,4.33,60.0,623.352045,190.112579,3333.000000
2025-04-28,3333.000000,33.165000,62.05,4.205,4.33,60.0,623.352045,137.450943,3333.600000
2025-04-29,3333.600000,33.577000,60.42,4.171,4.33,60.0,623.352045,107.072899,3319.100000



NaN count per column:
Gold_Futures             0
Silver_Futures           0
Crude_Oil_Futures        0
UST10Y_Treasury_Yield    0
Federal_Funds_Rate       0
Employment_Pop_Ratio     0
gepu                     0
gpr_daily                0
target_t_plus_1          0
dtype: int64


In [ ]:
# =========================
# 5) Chronological Outer Split: 80% Train, 20% Test
# =========================
outer_train_size = int(len(df) * OUTER_TRAIN_RATIO)

df_train_outer = df.iloc[:outer_train_size].copy().reset_index(drop=True)
df_test = df.iloc[outer_train_size:].copy().reset_index(drop=True)

print("df_train_outer shape:", df_train_outer.shape)
print("df_test shape:", df_test.shape)
print()
print("Outer-train date span / index span:", df_train_outer.index.min(), "to", df_train_outer.index.max())
print("Test date span / index span:", df_test.index.min(), "to", df_test.index.max())

df_train_outer shape: (2048, 9)
df_test shape: (513, 9)

Outer-train date span / index span: 0 to 2047
Test date span / index span: 0 to 512


In [ ]:
# =========================
# 6) Feature / Target Split
# =========================
feature_cols = [col for col in df_train_outer.columns if col != MODEL_TARGET_COL]
print("Number of features:", len(feature_cols))
print(feature_cols)

X_train_outer_raw = df_train_outer[feature_cols].copy()
y_train_outer_raw = df_train_outer[[MODEL_TARGET_COL]].copy()

X_test_raw = df_test[feature_cols].copy()
y_test_raw = df_test[[MODEL_TARGET_COL]].copy()

Number of features: 8
['Gold_Futures', 'Silver_Futures', 'Crude_Oil_Futures', 'UST10Y_Treasury_Yield', 'Federal_Funds_Rate', 'Employment_Pop_Ratio', 'gepu', 'gpr_daily']


In [ ]:
# =========================
# 7) Sequence Builder
# =========================
def create_sequences(X_df, y_df, lookback):
    X_values = X_df.values
    y_values = y_df.values.reshape(-1)

    X_seq, y_seq = [], []
    for i in range(lookback, len(X_df)):
        X_seq.append(X_values[i - lookback:i, :])
        y_seq.append(y_values[i])

    return np.array(X_seq), np.array(y_seq)


def create_sequences_with_context(X_prev, y_prev, X_block, y_block, lookback):
    X_full = pd.concat([X_prev.tail(lookback), X_block], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_prev.tail(lookback), y_block], axis=0).reset_index(drop=True)
    return create_sequences(X_full, y_full, lookback)

In [ ]:
# =========================
# 9) Model Builder
# =========================
def build_cnn_bilstm(input_shape, params):
    dropout_rate = float(params["dropout_rate"])
    l2_reg = float(params.get("l2_reg", 1.0e-5))
    recurrent_dropout = min(0.12, dropout_rate / 2.0)

    model = Sequential([
        Input(shape=input_shape),

        # Short windows (lookback ~10-18) usually do not benefit from max-pooling.
        # Preserve temporal resolution and use causal padding for one-step-ahead forecasting.
        Conv1D(
            filters=params["filters"],
            kernel_size=params["kernel_size"],
            activation="relu",
            padding="causal",
            kernel_regularizer=l2(l2_reg)
        ),
        BatchNormalization(),
        SpatialDropout1D(dropout_rate),

        Bidirectional(
            LSTM(
                params["lstm_units"],
                activation="tanh",
                recurrent_dropout=recurrent_dropout,
                return_sequences=True,
                kernel_regularizer=l2(l2_reg),
                recurrent_regularizer=l2(l2_reg)
            )
        ),
        Dropout(dropout_rate),

        Bidirectional(
            LSTM(
                max(16, params["lstm_units"] // 2),
                activation="tanh",
                recurrent_dropout=recurrent_dropout,
                return_sequences=False,
                kernel_regularizer=l2(l2_reg),
                recurrent_regularizer=l2(l2_reg)
            )
        ),
        Dropout(dropout_rate),

        Dense(
            params["dense_units"],
            activation="relu",
            kernel_regularizer=l2(l2_reg)
        ),
        Dropout(dropout_rate),

        Dense(1)
    ])

    model.compile(
        optimizer=Adam(
            learning_rate=params["learning_rate"],
            clipnorm=1.0
        ),
        loss=tf.keras.losses.Huber(),
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model


In [ ]:
optuna_trial_logs = []

def optuna_objective(trial):
    set_global_seed(TUNING_SEED)

    params = {
        "lookback": trial.suggest_int(
            "lookback",
            int(PBOUNDS["lookback"][0]),
            int(PBOUNDS["lookback"][1])
        ),

        "filters": trial.suggest_int(
            "filters",
            int(PBOUNDS["filters"][0]),
            int(PBOUNDS["filters"][1])
        ),

        "kernel_size": trial.suggest_int(
            "kernel_size",
            int(PBOUNDS["kernel_size"][0]),
            int(PBOUNDS["kernel_size"][1])
        ),

        "lstm_units": trial.suggest_int(
            "lstm_units",
            int(PBOUNDS["lstm_units"][0]),
            int(PBOUNDS["lstm_units"][1])
        ),

        "dense_units": trial.suggest_int(
            "dense_units",
            int(PBOUNDS["dense_units"][0]),
            int(PBOUNDS["dense_units"][1])
        ),

        "dropout_rate": trial.suggest_float(
            "dropout_rate",
            float(PBOUNDS["dropout_rate"][0]),
            float(PBOUNDS["dropout_rate"][1])
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            float(PBOUNDS["learning_rate"][0]),
            float(PBOUNDS["learning_rate"][1]),
            log=True
        ),

        "l2_reg": trial.suggest_float(
            "l2_reg",
            float(PBOUNDS["l2_reg"][0]),
            float(PBOUNDS["l2_reg"][1]),
            log=True
        ),

        "batch_size_exp": trial.suggest_int(
            "batch_size_exp",
            int(PBOUNDS["batch_size_exp"][0]),
            int(PBOUNDS["batch_size_exp"][1])
        ),
    }

    params["batch_size"] = int(2 ** params["batch_size_exp"])

    # Safety clamps
    params["lookback"] = max(2, params["lookback"])
    params["filters"] = max(8, params["filters"])
    params["kernel_size"] = max(1, params["kernel_size"])
    params["lstm_units"] = max(8, params["lstm_units"])
    params["dense_units"] = max(4, params["dense_units"])
    params["batch_size"] = max(8, params["batch_size"])

    tscv = TimeSeriesSplit(n_splits=N_SPLITS_WALK_FORWARD)
    fold_rmses = []

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(df_train_outer), start=1):
        fold_train_raw = df_train_outer.iloc[train_idx].copy().reset_index(drop=True)
        fold_val_raw = df_train_outer.iloc[val_idx].copy().reset_index(drop=True)

        X_fold_train_raw = fold_train_raw[feature_cols].copy()
        y_fold_train_raw = fold_train_raw[[MODEL_TARGET_COL]].copy()

        X_fold_val_raw = fold_val_raw[feature_cols].copy()
        y_fold_val_raw = fold_val_raw[[MODEL_TARGET_COL]].copy()

        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()

        X_fold_train_scaled = pd.DataFrame(
            scaler_X.fit_transform(X_fold_train_raw),
            columns=feature_cols
        )
        X_fold_val_scaled = pd.DataFrame(
            scaler_X.transform(X_fold_val_raw),
            columns=feature_cols
        )

        y_fold_train_scaled = pd.DataFrame(
            scaler_y.fit_transform(y_fold_train_raw),
            columns=[MODEL_TARGET_COL]
        )
        y_fold_val_scaled = pd.DataFrame(
            scaler_y.transform(y_fold_val_raw),
            columns=[MODEL_TARGET_COL]
        )

        X_train_seq, y_train_seq = create_sequences(
            X_fold_train_scaled,
            y_fold_train_scaled,
            params["lookback"]
        )

        X_val_seq, y_val_seq = create_sequences_with_context(
            X_fold_train_scaled,
            y_fold_train_scaled,
            X_fold_val_scaled,
            y_fold_val_scaled,
            params["lookback"]
        )

        if len(X_train_seq) == 0 or len(X_val_seq) == 0:
            return float("inf")

        model = build_cnn_bilstm(
            input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
            params=params
        )

        es = EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            min_delta=EARLY_STOPPING_MIN_DELTA,
            restore_best_weights=True,
            verbose=0
        )

        rlr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=0
        )

        model.fit(
            X_train_seq,
            y_train_seq,
            validation_data=(X_val_seq, y_val_seq),
            epochs=MAX_EPOCHS_TUNING,
            batch_size=params["batch_size"],
            shuffle=False,
            verbose=0,
            callbacks=[es, rlr]
        )

        y_val_pred_scaled = model.predict(X_val_seq, verbose=0).reshape(-1, 1)
        y_val_true_scaled = y_val_seq.reshape(-1, 1)

        y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled).reshape(-1)
        y_val_true = scaler_y.inverse_transform(y_val_true_scaled).reshape(-1)

        fold_rmse = np.sqrt(mean_squared_error(y_val_true, y_val_pred))
        fold_rmses.append(fold_rmse)

        trial.report(float(fold_rmse), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_rmse = float(np.mean(fold_rmses))

    optuna_trial_logs.append({
        "trial_number": trial.number,
        "params": params.copy(),
        "fold_rmses": fold_rmses.copy(),
        "mean_rmse": mean_rmse,
    })

    return mean_rmse


In [ ]:
set_global_seed(TUNING_SEED)

sampler = TPESampler(seed=TUNING_SEED)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=max(5, INIT_POINTS),
    n_warmup_steps=2
)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    study_name="cnn_bilstm_walkforward_optuna"
)

study.optimize(
    optuna_objective,
    n_trials=INIT_POINTS + N_ITER,
    show_progress_bar=False
)

print("Best Optuna result:")
print("Best trial number:", study.best_trial.number)
print("Best objective (mean RMSE):", study.best_value)
print("Best params:", study.best_trial.params)

[I 2026-04-06 16:00:30,077] A new study created in memory with name: cnn_bilstm_walkforward_optuna
[I 2026-04-06 16:12:00,973] Trial 0 finished with value: 298.5475565576878 and parameters: {'lookback': 13, 'filters': 39, 'kernel_size': 4, 'lstm_units': 43, 'dense_units': 10, 'dropout_rate': 0.13559945203362025, 'learning_rate': 8.484143720180103e-05, 'l2_reg': 0.00021766241123453658, 'batch_size_exp': 4}. Best is trial 0 with value: 298.5475565576878.
[I 2026-04-06 16:18:54,108] Trial 1 finished with value: 288.0918783559372 and parameters: {'lookback': 15, 'filters': 33, 'kernel_size': 2, 'lstm_units': 56, 'dense_units': 22, 'dropout_rate': 0.14123391106782762, 'learning_rate': 9.615494850936552e-05, 'l2_reg': 3.1261029103110603e-06, 'batch_size_exp': 4}. Best is trial 1 with value: 288.0918783559372.
[I 2026-04-06 16:26:58,705] Trial 2 finished with value: 242.86944838077903 and parameters: {'lookback': 12, 'filters': 29, 'kernel_size': 3, 'lstm_units': 33, 'dense_units': 18, 'dropo

In [ ]:
# =========================
# 11) Extract Best Hyperparameters
# =========================
best_raw = study.best_trial.params

best_params = {
    "lookback": int(best_raw["lookback"]),
    "filters": int(best_raw["filters"]),
    "kernel_size": int(best_raw["kernel_size"]),
    "lstm_units": int(best_raw["lstm_units"]),
    "dense_units": int(best_raw["dense_units"]),
    "dropout_rate": float(best_raw["dropout_rate"]),
    "learning_rate": float(best_raw["learning_rate"]),
    "l2_reg": float(best_raw["l2_reg"]),
    "batch_size": int(2 ** int(best_raw["batch_size_exp"])),
}

print("Best hyperparameters after conversion:")
print(best_params)


In [ ]:
# =========================
# 12) Report Per-Trial Per-Fold RMSE from Optuna Walk-Forward
# =========================
optuna_report_rows = []

print("=" * 100)
print("OPTUNA WALK-FORWARD TUNING REPORT")
print("=" * 100)

for log_idx, trial in enumerate(sorted(optuna_trial_logs, key=lambda x: x["trial_number"]), start=1):
    row = {
        "logged_trial_index": log_idx,
        "trial_number": trial["trial_number"],
        "mean_rmse": trial["mean_rmse"],
    }
    print(f"Trial {trial['trial_number']}")
    print("Params:", trial["params"])
    for fold_i, rmse in enumerate(trial["fold_rmses"], start=1):
        print(f"  Fold {fold_i} RMSE = {rmse:.6f}")
        row[f"fold_{fold_i}_rmse"] = rmse
    print(f"  Mean RMSE = {trial['mean_rmse']:.6f}")
    print("-" * 100)
    optuna_report_rows.append(row)

bayes_report_df = pd.DataFrame(optuna_report_rows)
display(bayes_report_df)


In [ ]:
# =========================
# 13) Final Raw Split Before Scaling (Leakage-Free Epoch Selection)
# =========================
n_train_outer_raw = len(df_train_outer)
n_val_raw_final = max(1, int(n_train_outer_raw * FINAL_VAL_RATIO_WITHIN_OUTER_TRAIN))
n_fit_raw_final = n_train_outer_raw - n_val_raw_final

df_fit_final_raw = df_train_outer.iloc[:n_fit_raw_final].copy().reset_index(drop=True)
df_val_final_raw = df_train_outer.iloc[n_fit_raw_final:].copy().reset_index(drop=True)

X_fit_final_raw = df_fit_final_raw[feature_cols].copy()
y_fit_final_raw = df_fit_final_raw[[MODEL_TARGET_COL]].copy()

X_val_final_raw = df_val_final_raw[feature_cols].copy()
y_val_final_raw = df_val_final_raw[[MODEL_TARGET_COL]].copy()

# Fit scalers on final-fit only so the internal validation stays leakage-free
scaler_X_epoch = MinMaxScaler()
scaler_y_epoch = MinMaxScaler()

X_fit_final_scaled = pd.DataFrame(
    scaler_X_epoch.fit_transform(X_fit_final_raw),
    columns=feature_cols
)

y_fit_final_scaled = pd.DataFrame(
    scaler_y_epoch.fit_transform(y_fit_final_raw),
    columns=[MODEL_TARGET_COL]
)

X_val_final_scaled = pd.DataFrame(
    scaler_X_epoch.transform(X_val_final_raw),
    columns=feature_cols
)

y_val_final_scaled = pd.DataFrame(
    scaler_y_epoch.transform(y_val_final_raw),
    columns=[MODEL_TARGET_COL]
)

print("Leakage-free final raw split:")
print("df_fit_final_raw shape:", df_fit_final_raw.shape)
print("df_val_final_raw shape:", df_val_final_raw.shape)
print("Scaled final-fit shape:", X_fit_final_scaled.shape, y_fit_final_scaled.shape)
print("Scaled final-val shape:", X_val_final_scaled.shape, y_val_final_scaled.shape)


In [ ]:
# =========================
# 14) Create Leakage-Free Final Fit/Val Sequences + Refit/Test Sequences
# =========================
LOOKBACK = best_params["lookback"]

# Sequences used only to choose best_epoch in a leakage-free way
X_fit_final, y_fit_final = create_sequences(
    X_fit_final_scaled,
    y_fit_final_scaled,
    LOOKBACK
)

X_val_final, y_val_final = create_sequences_with_context(
    X_fit_final_scaled,
    y_fit_final_scaled,
    X_val_final_scaled,
    y_val_final_scaled,
    LOOKBACK
)

print("Leakage-free final fit sequences:", X_fit_final.shape, y_fit_final.shape)
print("Leakage-free final validation sequences:", X_val_final.shape, y_val_final.shape)

# Refit scalers on the full outer-train only AFTER epoch selection setup is defined.
# These are used for the final full-data refit and untouched test evaluation.
scaler_X_final = MinMaxScaler()
scaler_y_final = MinMaxScaler()

X_train_outer_scaled = pd.DataFrame(
    scaler_X_final.fit_transform(X_train_outer_raw),
    columns=feature_cols
)

y_train_outer_scaled = pd.DataFrame(
    scaler_y_final.fit_transform(y_train_outer_raw),
    columns=[MODEL_TARGET_COL]
)

X_test_scaled = pd.DataFrame(
    scaler_X_final.transform(X_test_raw),
    columns=feature_cols
)

y_test_scaled = pd.DataFrame(
    scaler_y_final.transform(y_test_raw),
    columns=[MODEL_TARGET_COL]
)

X_train_outer_seq, y_train_outer_seq = create_sequences(
    X_train_outer_scaled,
    y_train_outer_scaled,
    LOOKBACK
)

X_test_seq, y_test_seq = create_sequences_with_context(
    X_train_outer_scaled,
    y_train_outer_scaled,
    X_test_scaled,
    y_test_scaled,
    LOOKBACK
)

print("Final full outer-train sequences for refit:", X_train_outer_seq.shape, y_train_outer_seq.shape)
print("Final test sequences with train-tail context:", X_test_seq.shape, y_test_seq.shape)


In [ ]:
# =========================
# 15) Final Evaluation Setup Check
# =========================
print("Epoch selection uses fit-only scalers and train-tail context for validation.")
print("Final refit uses all outer-train data.")
print("Test evaluation uses train-tail context from the end of outer-train.")


In [ ]:
# =========================
# 16) Save / Load Helpers for Inference
# =========================
import json
import os
import pickle

def save_forecasting_artifacts(
    model,
    save_dir: str,
    x_scaler,
    feature_cols,
    target_col: str,
    lookback: int,
    y_scaler=None,
    model_name: str = "final_model.keras",
    extra_metadata: dict | None = None,
) -> None:
    os.makedirs(save_dir, exist_ok=True)

    # Save trained Keras model
    model.save(os.path.join(save_dir, model_name))

    # Save scalers
    with open(os.path.join(save_dir, "x_scaler.pkl"), "wb") as f:
        pickle.dump(x_scaler, f)

    if y_scaler is not None:
        with open(os.path.join(save_dir, "y_scaler.pkl"), "wb") as f:
            pickle.dump(y_scaler, f)

    # Save metadata
    metadata = {
        "feature_cols": list(feature_cols),
        "target_col": target_col,
        "lookback": int(lookback),
        "model_file": model_name,
        "has_y_scaler": y_scaler is not None,
    }
    if extra_metadata is not None:
        metadata.update(extra_metadata)

    with open(os.path.join(save_dir, "model_metadata.pkl"), "wb") as f:
        pickle.dump(metadata, f)

    with open(os.path.join(save_dir, "model_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

def load_forecasting_artifacts(save_dir: str):
    with open(os.path.join(save_dir, "model_metadata.pkl"), "rb") as f:
        metadata = pickle.load(f)

    model = tf.keras.models.load_model(os.path.join(save_dir, metadata["model_file"]))

    with open(os.path.join(save_dir, "x_scaler.pkl"), "rb") as f:
        x_scaler = pickle.load(f)

    y_scaler = None
    if metadata.get("has_y_scaler", False):
        with open(os.path.join(save_dir, "y_scaler.pkl"), "rb") as f:
            y_scaler = pickle.load(f)

    return model, x_scaler, y_scaler, metadata

os.makedirs(SAVE_ROOT_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
print(f"Model save root: {SAVE_ROOT_DIR}")
print(f"Reports dir: {REPORTS_DIR}")


In [ ]:
# =========================
# 17) Train Final Models Across Seeds, Select best_epoch, Refit on Full Outer-Train, Evaluate, and Save ALL Seed Models
# =========================
final_results = []
seed_histories = {}
seed_predictions = {}

for seed in FINAL_SEEDS:
    print("\n" + "=" * 100)
    print(f"SEED {seed}")
    print("=" * 100)

    set_global_seed(seed)

    # -----------------------------------------------------------------
    # Phase A: choose best_epoch using leakage-free final fit/validation
    # -----------------------------------------------------------------
    selection_model = build_cnn_bilstm(
        input_shape=(X_fit_final.shape[1], X_fit_final.shape[2]),
        params=best_params
    )

    es = EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
        restore_best_weights=True,
        verbose=0
    )

    rlr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=0
    )

    history = selection_model.fit(
        X_fit_final,
        y_fit_final,
        validation_data=(X_val_final, y_val_final),
        epochs=MAX_EPOCHS_FINAL,
        batch_size=best_params["batch_size"],
        shuffle=False,
        verbose=0,
        callbacks=[es, rlr]
    )

    seed_histories[seed] = history

    print(f"\nPer-epoch loss and val_loss for seed {seed}:")
    print("epoch\tloss\t\tval_loss")
    for epoch_idx, (loss_value, val_loss_value) in enumerate(
        zip(history.history["loss"], history.history["val_loss"]), start=1
    ):
        print(f"{epoch_idx}\t{loss_value:.8f}\t{val_loss_value:.8f}")

    best_epoch_idx = int(np.argmin(history.history["val_loss"]))
    best_epoch = best_epoch_idx + 1
    best_loss = float(history.history["loss"][best_epoch_idx])
    best_val_loss = float(history.history["val_loss"][best_epoch_idx])

    # -----------------------------------------------------------------
    # Phase B: refit from scratch on the FULL outer-train for best_epoch
    # -----------------------------------------------------------------
    set_global_seed(seed)

    refit_model = build_cnn_bilstm(
        input_shape=(X_train_outer_seq.shape[1], X_train_outer_seq.shape[2]),
        params=best_params
    )

    refit_model.fit(
        X_train_outer_seq,
        y_train_outer_seq,
        epochs=best_epoch,
        batch_size=best_params["batch_size"],
        shuffle=False,
        verbose=0
    )

    # Evaluate on untouched outer test using train-tail context
    y_pred_test_scaled = refit_model.predict(X_test_seq, verbose=0).reshape(-1, 1)
    y_true_test_scaled = y_test_seq.reshape(-1, 1)

    y_pred_test = scaler_y_final.inverse_transform(y_pred_test_scaled).reshape(-1)
    y_true_test = scaler_y_final.inverse_transform(y_true_test_scaled).reshape(-1)

    rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    r2 = r2_score(y_true_test, y_pred_test)

    print(
        f"\nSeed {seed} -> best_epoch={best_epoch}, "
        f"best_loss={best_loss:.8f}, best_val_loss={best_val_loss:.8f}, "
        f"RMSE={rmse:.6f}, R²={r2:.6f}"
    )

    final_results.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "best_loss": best_loss,
        "best_val_loss": best_val_loss,
        "rmse_test": rmse,
        "r2_test": r2,
    })

    seed_predictions[seed] = {
        "y_true_test": y_true_test,
        "y_pred_test": y_pred_test,
    }

    # Save every trained seed model with all inference artifacts
    if SAVE_ALL_SEED_MODELS:
        seed_dir = os.path.join(SAVE_ROOT_DIR, f"seed_{seed}")
        save_forecasting_artifacts(
            model=refit_model,
            save_dir=seed_dir,
            x_scaler=scaler_X_final,
            y_scaler=scaler_y_final,
            feature_cols=feature_cols,
            target_col=MODEL_TARGET_COL,
            lookback=best_params["lookback"],
            model_name=f"cnn_bilstm_seed{seed}.keras",
            extra_metadata={
                "seed": seed,
                "horizon": HORIZON,
                "best_params": best_params,
                "rmse_test": float(rmse),
                "r2_test": float(r2),
                "best_epoch": int(best_epoch),
            },
        )
        print(f"Saved seed {seed} artifacts to: {seed_dir}")


In [ ]:
# =========================
# 18) Per-Seed RMSE and R² Report
# =========================
final_results_df = pd.DataFrame(final_results)
print("=" * 100)
print("PER-SEED FINAL TEST RESULTS")
print("=" * 100)
display(final_results_df)

rmse_mean = final_results_df["rmse_test"].mean()
rmse_std = final_results_df["rmse_test"].std(ddof=1)

r2_mean = final_results_df["r2_test"].mean()
r2_std = final_results_df["r2_test"].std(ddof=1)

loss_mean = final_results_df["best_loss"].mean()
loss_std = final_results_df["best_loss"].std(ddof=1)

val_loss_mean = final_results_df["best_val_loss"].mean()
val_loss_std = final_results_df["best_val_loss"].std(ddof=1)

print("\nSUMMARY ACROSS SEEDS")
print(f"RMSE  = {rmse_mean:.6f} ± {rmse_std:.6f}")
print(f"R²    = {r2_mean:.6f} ± {r2_std:.6f}")
print(f"Loss  = {loss_mean:.8f} ± {loss_std:.8f}")
print(f"Val loss = {val_loss_mean:.8f} ± {val_loss_std:.8f}")

In [ ]:
# =========================
# 19) Plot Every Per-Seed Loss vs Val_Loss
# =========================
for seed in FINAL_SEEDS:
    history = seed_histories[seed]

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(f"Seed {seed} - Training Loss vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# =========================
# 20) Plot Every Per-Seed Predicted vs Actual on df_test
# =========================
for seed in FINAL_SEEDS:
    y_true_test = seed_predictions[seed]["y_true_test"]
    y_pred_test = seed_predictions[seed]["y_pred_test"]

    rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    r2 = r2_score(y_true_test, y_pred_test)

    plt.figure(figsize=(10, 4))
    plt.plot(y_true_test, label="Actual")
    plt.plot(y_pred_test, label="Predicted", linestyle="--")
    plt.title(f"Seed {seed} - Test Actual vs Predicted | RMSE={rmse:.4f}, R²={r2:.4f}")
    plt.xlabel("Test Sequence Index")
    plt.ylabel("Target")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# =========================
# 21) Save Final Reports
# =========================
final_results_path = os.path.join(REPORTS_DIR, "gold_final_seed_results_optimized.csv")
optuna_report_path = os.path.join(REPORTS_DIR, "gold_optuna_walkforward_report_optimized.csv")
best_params_path = os.path.join(REPORTS_DIR, "gold_best_params_optimized.json")

bayes_report_df.to_csv(optuna_report_path, index=False)
final_results_df.to_csv(final_results_path, index=False)

with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=4)

print("Saved:")
print(f"- {optuna_report_path}")
print(f"- {final_results_path}")
print(f"- {best_params_path}")
print(f"- {SAVE_ROOT_DIR}/seed_*/")
